# Customer Churn — Visual Storytelling

**Audience:** retention / customer-success leadership (non-technical).
**Goal:** turn the findings validated in [`01_eda.ipynb`](01_eda.ipynb) into an *interactive* business narrative — **not** a re-analysis.

> **Why a second notebook?** The EDA used Seaborn/matplotlib for *analytical* exploration. This one uses **Plotly** for *communication*: interactive, hover-rich charts a stakeholder can explore. Each section answers one business question. No new statistics are computed — only the numbers the EDA already justified.

**Design principles used throughout:**
- One chart = one message (the title states the takeaway, not the variable).
- Consistent color language: 🔴 red = at-risk/churn, 🔵 blue = retained.
- A gray dashed baseline (26.6% overall churn) on every rate chart, so "high" and "low" are always relative to the average.

## 0. Setup — data prep + a reusable theme

We replicate the *exact* cleaning decided in the EDA (drop the 11 brand-new `tenure=0` customers, cast `TotalCharges`, encode the target) and fix a single color/template language for the whole notebook. Defining the palette once is what keeps the story visually coherent.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# --- One visual language for the whole notebook ---
pio.templates.default = "plotly_white"   # clean background, good for slides
pio.renderers.default = "plotly_mimetype+notebook_connected"  # also emit HTML so charts render on nbviewer
CHURN_COLOR  = "#E45756"   # red  -> at-risk / churned
RETAIN_COLOR = "#4C78A8"   # blue -> retained
SEQ = px.colors.sequential.Reds          # sequential scale for "more red = more churn"

# --- Exact data prep validated in 01_eda.ipynb ---
df = pd.read_csv("../data/subscribers.csv")
df = df[df["TotalCharges"] != " "]                 # 11 brand-new customers, no billing cycle yet
df["TotalCharges"] = df["TotalCharges"].astype(float)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})  # 1 = churned, 0 = retained

BASELINE = df["Churn"].mean()                       # overall churn rate -> our reference line
print(f"{len(df):,} subscribers loaded · baseline churn rate = {BASELINE:.1%}")

7,032 subscribers loaded · baseline churn rate = 26.6%


## 1. The problem in one number

Every executive story opens with the headline. Before *who* or *why*, the audience needs the *how big*.

In [2]:
churned, total = int(df["Churn"].sum()), len(df)

fig = go.Figure(go.Indicator(
    mode="number",
    value=BASELINE * 100,
    number={"suffix": "%", "font": {"size": 72, "color": CHURN_COLOR}},
    title={"text": "Overall churn rate"
                   f"<br><span style='font-size:0.55em;color:gray'>"
                   f"{churned:,} of {total:,} subscribers cancelled</span>"},
))
fig.update_layout(height=320, margin=dict(t=90, b=20))
fig.show()

**Design note — why a KPI indicator, not a chart?** A single number has the highest *data-ink-to-message* ratio when the message *is* the number. `go.Indicator` is Plotly's purpose-built widget for a headline metric — it anchors the whole narrative before we break it down.

## 2. Who churns most? Contract type is the dominant driver

The EDA ranked `Contract` as the single strongest predictor. We lead with it.

In [3]:
contract_churn = (
    df.groupby("Contract")["Churn"].mean()
      .reindex(["Month-to-month", "One year", "Two year"])  # logical order, not alphabetical
      .reset_index()
)

fig = px.bar(
    contract_churn, x="Contract", y="Churn",
    color="Churn", color_continuous_scale=SEQ,
    text=contract_churn["Churn"].map("{:.0%}".format),
    title="Month-to-month subscribers churn ~15× more than 2-year contracts",
)
fig.add_hline(y=BASELINE, line_dash="dash", line_color="gray",
              annotation_text=f"baseline {BASELINE:.0%}", annotation_position="top right")
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_tickformat=".0%", yaxis_title="Churn rate",
                  xaxis_title="", coloraxis_showscale=False, height=420)
fig.show()
fig.write_image("../images/01_churn_by_contract.png", scale=2)  # static copy for the README (GitHub can't render Plotly)

**Design note — ordered bars + encoded color.** Categories are reordered by commitment level (month → year → two-year), never alphabetically, so the eye reads a trend. Color *redundantly* encodes the same value as height — redundancy is fine when it reinforces the one message ("redder = riskier"). Direct value labels remove the need to read the axis.

## 3. The danger window: churn risk is front-loaded

The EDA showed churners have ~4× lower tenure. A rate-by-tenure curve turns that static fact into the *actionable* shape: **when** in the lifecycle the risk lives.

In [4]:
bins   = [0, 3, 6, 12, 24, 48, 72]
labels = ["0–3", "3–6", "6–12", "12–24", "24–48", "48–72"]
df["tenure_bucket"] = pd.cut(df["tenure"], bins=bins, labels=labels, include_lowest=True)

tenure_churn = (
    df.groupby("tenure_bucket", observed=True)["Churn"]
      .mean().reset_index()
)

fig = px.line(
    tenure_churn, x="tenure_bucket", y="Churn", markers=True,
    title="The first 6 months are the danger window — churn risk decays with tenure",
)
fig.add_hline(y=BASELINE, line_dash="dash", line_color="gray",
              annotation_text=f"baseline {BASELINE:.0%}", annotation_position="bottom right")
fig.update_traces(line_color=CHURN_COLOR, line_width=3, marker_size=9)
fig.update_layout(yaxis_tickformat=".0%", yaxis_title="Churn rate",
                  xaxis_title="Tenure (months)", hovermode="x unified", height=420)
fig.show()
fig.write_image("../images/02_churn_by_tenure.png", scale=2)  # static copy for the README (GitHub can't render Plotly)

**Design note — a line for an ordered continuum.** Tenure buckets are *ordinal*, so a line (not bars) is correct: it implies the progression between points. `hovermode='x unified'` lets a stakeholder scrub across the curve and read each bucket — the kind of exploration a static EDA chart can't offer. **Business read:** retention budget should concentrate on the 0–6 month onboarding window.

## 4. The 'less-committed' cluster

The EDA flagged that `Month-to-month` + `Electronic check` + `Fiber optic` might not be three independent signals but one *profile*. A sunburst lets us see how these nest and where churn concentrates.

In [5]:
seg = (
    df.groupby(["Contract", "PaymentMethod", "InternetService"], observed=True)
      .agg(subscribers=("Churn", "size"), churn_rate=("Churn", "mean"))
      .reset_index()
)

fig = px.sunburst(
    seg, path=["Contract", "PaymentMethod", "InternetService"],
    values="subscribers", color="churn_rate", color_continuous_scale=SEQ,
    title="Where churn concentrates: drill from contract → payment → internet",
)
fig.update_layout(height=560, coloraxis_colorbar_title="churn<br>rate")
fig.update_traces(hovertemplate="<b>%{label}</b><br>subscribers: %{value}<br>churn rate: %{color:.0%}<extra></extra>")
fig.show()
fig.write_image("../images/03_risk_segments_sunburst.png", scale=2)  # static copy for the README (GitHub can't render Plotly)

**Design note — hierarchy + size + color in one view.** A sunburst encodes *three* things at once: ring = level of the hierarchy, arc size = number of subscribers, color = churn rate. Click a wedge to drill in. The reddest outer wedges concentrated under *Month-to-month* confirm the EDA's hunch: these aren't independent drivers but facets of one *less-committed* customer profile — important, because the modeling notebook shouldn't treat them as fully independent features.

## 5. The protection gap

The four support/protection add-ons all showed the same pattern in the EDA: absence ≈ high churn. A grouped bar makes the *with-vs-without* gap directly comparable across services. (We drop the `No internet service` group — those customers can't have these add-ons, so they're not part of the question.)

In [6]:
addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]
rows = []
for a in addons:
    g = df.groupby(a, observed=True)["Churn"].mean()
    for status in ("Yes", "No"):
        rows.append({"service": a, "has_service": status, "churn": g[status]})
prot = pd.DataFrame(rows)

fig = px.bar(
    prot, x="service", y="churn", color="has_service", barmode="group",
    color_discrete_map={"Yes": RETAIN_COLOR, "No": CHURN_COLOR},
    category_orders={"has_service": ["No", "Yes"]},
    text=prot["churn"].map("{:.0%}".format),
    title="Subscribers without protection add-ons churn ~2–3× more",
)
fig.add_hline(y=BASELINE, line_dash="dash", line_color="gray")
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_tickformat=".0%", yaxis_title="Churn rate", xaxis_title="",
                  legend_title="Has service?", height=440)
fig.show()
fig.write_image("../images/04_protection_gap.png", scale=2)  # static copy for the README (GitHub can't render Plotly)

**Design note — grouped bars for a paired comparison.** The question is *within each service, does having it change churn?* Grouping the two bars side by side puts the comparison where the eye needs it. Consistent color mapping (blue=has, red=lacks) means the reader learns the legend once and applies it to all four pairs. **Business read:** bundling these add-ons could be a retention lever — the modeling notebook will test whether they collapse into a single 'has protection' feature.

## 6. Where to act first: a prioritization matrix

Knowing *who* churns isn't enough for budgeting — leadership needs to weigh **risk × value × volume**. A bubble chart puts all three on one canvas so the priority segments fall in the top-right.

In [7]:
seg2 = (
    df.groupby(["Contract", "InternetService"], observed=True)
      .agg(subscribers=("Churn", "size"),
           churn_rate=("Churn", "mean"),
           avg_monthly=("MonthlyCharges", "mean"))
      .reset_index()
)
seg2["segment"] = seg2["Contract"] + " · " + seg2["InternetService"]

fig = px.scatter(
    seg2, x="churn_rate", y="avg_monthly", size="subscribers", color="Contract",
    hover_name="segment", size_max=60,
    title="Act first on the top-right: high churn × high monthly revenue × large volume",
)
fig.add_vline(x=BASELINE, line_dash="dash", line_color="gray",
              annotation_text=f"baseline {BASELINE:.0%}", annotation_position="top")
fig.update_layout(xaxis_tickformat=".0%", xaxis_title="Churn rate",
                  yaxis_title="Avg monthly charges ($)", height=520)
fig.show()

**Design note — a bubble chart trades simplicity for density.** It encodes four dimensions (x=risk, y=value, size=volume, color=contract). That's the *most* a single chart should carry — any more and it stops communicating. The dashed line splits above/below-average risk. **Business read:** the biggest red bubbles to the right of the line are *month-to-month fiber* segments — high churn, high revenue, and large enough to move the needle. That's where outreach pays back first.

## Takeaways for the next notebook

This story sets up `03_model.ipynb`:

1. **Lifecycle matters** — the tenure curve suggests `tenure` will be a top model feature.
2. **The 'less-committed' cluster is real** — the sunburst confirms `Contract`, `PaymentMethod`, `InternetService` are correlated, so we'll watch for redundancy when modeling.
3. **Protection add-ons move together** — candidate for a single engineered 'has-protection' feature.
4. **Priority targets identified** — month-to-month fiber segments; the model should score these well.

> *Next: build a classifier that turns these descriptive signals into a per-subscriber churn probability.*